# 실험 04: LCEL 조립과 심화 확장

## 1. 실험 제목과 목표
- **제목**: LangChain Expression Language (LCEL) 완전 정복
- **목표**: 파이프 연산자(`|`)를 활용하여 프롬프트, 모델, 파서를 레고 블록처럼 조립하는 기본기를 익힙니다. 나아가 병렬 처리(`RunnableParallel`), 데이터 전달(`RunnablePassthrough`), 동적 파라미터 제어(`.bind`), 장애 대비 폴백(`.with_fallbacks`) 등 실무 필수 심화 기법을 완벽히 마스터합니다.

## 2. 실험 개요
1. **기본 실험**: 수동 변수 할당 vs LCEL 기본 체인(`prompt | model | parser`) 비교
2. **심화 실험 1 (병렬 처리)**: `RunnableParallel`로 두 개의 체인을 동시에 돌리고 결과 합치기
3. **심화 실험 2 (데이터 패스)**: `RunnablePassthrough`로 중간에 외부 데이터(예: RAG 검색 결과) 스윽 끼워넣기
4. **심화 실험 3 (동적 제어)**: `.bind()`를 통해 실행 직전에 모델의 `stop` 단어나 `temperature` 동적 주입
5. **심화 실험 4 (장애 대비)**: `.with_fallbacks()`로 에러 발생 시 예비 체인(Plan B)으로 부드럽게 넘어가는 기법
6. **실패 케이스**: 파이프라인 입출력 타입 불일치로 인한 LCEL 체인 붕괴 시연

## 3. 라이브러리 import

In [1]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

## 4. 환경 변수 로드 및 모델 준비

In [2]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

## 5. 기본 실험: 수동 조립 vs LCEL 파이프라인
이전 노트북까지는 각 단계를 변수로 담아 넘겨주었지만, 이제 하나의 체인으로 묶습니다.

In [3]:
prompt = PromptTemplate.from_template("{topic}에 대한 짧은 농담 하나 해줘.")
parser = StrOutputParser()

print("=== [1. 수동 연결 방식] ===")
formatted_prompt = prompt.format(topic="인공지능")
ai_response = llm.invoke(formatted_prompt)
final_string = parser.parse(ai_response.content)
print(final_string)

print("\n=== [2. LCEL 체인 방식] ===")
# 마법의 파이프 연산자(|)를 사용하여 데이터 흐름을 직관적으로 조립합니다.
chain = prompt | llm | parser
print(chain.invoke({"topic": "인공지능"}))

print("\n-> 결과: 코드가 압도적으로 간결해졌으며, 데이터가 왼쪽에서 오른쪽으로 흐르는 것이 한눈에 보입니다.")

=== [1. 수동 연결 방식] ===
왜 인공지능은 해적을 싫어할까요? 

항상 "정확한 데이터를 원해!"라고 하니까요! ☠️😄

=== [2. LCEL 체인 방식] ===
왜 인공지능은 농담을 잘 못할까요?  

왜냐하면, 항상 "진지하게" 사고하거든요! 😄

-> 결과: 코드가 압도적으로 간결해졌으며, 데이터가 왼쪽에서 오른쪽으로 흐르는 것이 한눈에 보입니다.


## 6. 심화 실험 1: 병렬 처리 (RunnableParallel)
입력값 하나로 여러 개의 체인을 동시에 돌릴 수 있습니다. (비동기 처리되어 매우 빠릅니다)

In [4]:
joke_chain = PromptTemplate.from_template("{topic}에 대한 농담:") | llm | StrOutputParser()
poem_chain = PromptTemplate.from_template("{topic}에 대한 짧은 시:") | llm | StrOutputParser()

# 두 체인을 병렬로 묶습니다.
parallel_chain = RunnableParallel(joke=joke_chain, poem=poem_chain)

print("[RunnableParallel 실행]")
result = parallel_chain.invoke({"topic": "커피"})

print("\n--- 🤣 농담 파트 ---")
print(result["joke"])
print("\n--- 📝 시 파트 ---")
print(result["poem"])

[RunnableParallel 실행]

--- 🤣 농담 파트 ---
커피와 관련된 농담 하나 드릴게요!

왜 커피는 항상 사람들을 행복하게 할까요?

커피는 항상 "안녕하세요!"라고 인사를 하니까요! (에스프레소, 'Espresso!') 

즐거운 하루 되세요! ☕️

--- 📝 시 파트 ---
커피 한 잔, 따뜻한 손길  
어두운 아침에 빛을 주네  
은은한 향기, 마음을 감싸  
시간이 멈춘 듯, 여유로워  

작은 모금에 세상이 담기고  
고단한 하루도 잊게 하네  
커피 한 잔, 나의 작은 기쁨  
추억을 안고, 다시 시작해.


## 7. 심화 실험 2: 데이터 패스 (RunnablePassthrough)
입력 딕셔너리에 사용자가 입력하지 않은 데이터(예: 외부 검색 결과)를 중간에 스윽 추가해 넘겨주는 RAG의 핵심 기법입니다.

In [5]:
# 가짜 검색 함수 (실제로는 VectorDB에서 문서를 가져오는 로직이 들어갑니다)
def fake_retriever(query):
    return "[검색 결과: 토끼는 당근을 좋아하지만, 사실 건초를 주식으로 먹어야 건강합니다.]"

rag_prompt = PromptTemplate.from_template(
    "참고 자료: {context}\n\n질문: {question}\n\n참고 자료를 바탕으로 질문에 답하세요."
)

# 1. 사용자는 {'question': '...'} 만 입력합니다.
# 2. RunnablePassthrough가 question을 복사해 보존하고, 
# 3. 동시에 question을 fake_retriever에 넣어 context를 알아서 생성합니다.
rag_chain = (
    {"context": itemgetter("question") | RunnableLambda(fake_retriever), "question": RunnablePassthrough()}
    if False else  # 간략화된 Passthrough 문법을 사용합니다.
    {"context": lambda x: fake_retriever(x["question"]), "question": RunnablePassthrough()} 
    | rag_prompt 
    | llm 
    | StrOutputParser()
)

print("[RunnablePassthrough RAG 체인]")
print(rag_chain.invoke({"question": "토끼는 뭐 먹고 살아?"}))

[RunnablePassthrough RAG 체인]
토끼는 당근을 좋아하지만, 사실 건강을 위해 건초를 주식으로 먹어야 합니다. 따라서 토끼는 주로 건초를 먹고 살아야 합니다.


## 8. 심화 실험 3: 동적 파라미터 제어 (.bind)
체인을 미리 조립해두었는데, 특정 상황에서는 모델이 특정 단어를 말하면 출력을 멈추게(stop) 하고 싶을 때 사용합니다.

In [6]:
prompt_bind = PromptTemplate.from_template("{count}부터 1까지 거꾸로 세어봐.")

# 기본 체인
chain_normal = prompt_bind | llm | StrOutputParser()

# .bind()를 사용해 특정 조건에서 멈추도록 강제 주입
# LLM이 "3"이라는 글자를 생성하려는 순간 강제로 생성을 종료시킵니다.
chain_stopped = prompt_bind | llm.bind(stop=["3"]) | StrOutputParser()

print("=== [기본 실행] ===")
print(chain_normal.invoke({"count": 5}))

print("\n=== [.bind(stop=['3']) 실행] ===")
print(chain_stopped.invoke({"count": 5}))
print("-> 결과: 3이 출력되기 직전에 출력이 가위로 잘린 것처럼 멈춥니다!")

=== [기본 실행] ===
물론입니다! 5부터 1까지 거꾸로 세어보면 다음과 같습니다:

5, 4, 3, 2, 1.

=== [.bind(stop=['3']) 실행] ===
물론입니다! 5부터 1까지 거꾸로 세어보면 다음과 같습니다:

5, 4, 
-> 결과: 3이 출력되기 직전에 출력이 가위로 잘린 것처럼 멈춥니다!


## 9. 심화 실험 4: 장애 대비 폴백 (Fallbacks)
메인 모델이나 파서가 에러를 뿜었을 때, 서비스가 죽지 않도록 예비 플랜(Plan B)을 마련합니다.

In [7]:
# 일부러 무조건 에러를 내는 망가진 모델 시뮬레이션
from langchain_core.runnables import RunnableLambda
def broken_model(x):
    raise ValueError("API Rate Limit Exceeded (가짜 에러)!!")

bad_chain = PromptTemplate.from_template("{text}") | RunnableLambda(broken_model)

# 예비 체인 (정상 모델)
backup_chain = PromptTemplate.from_template("문제가 생겨 백업 모델이 응답합니다: {text}") | llm | StrOutputParser()

# 메인 체인에 백업 체인을 폴백으로 연결합니다.
safe_chain = bad_chain.with_fallbacks([backup_chain])

print("[Fallback 체인 실행]")
print(safe_chain.invoke({"text": "안녕하세요!"}))
print("\n-> 결과: 메인 체인에서 치명적인 에러가 났지만 서비스가 죽지 않고 백업 체인으로 부드럽게 넘어갔습니다.")

[Fallback 체인 실행]
안녕하세요! 어떻게 도와드릴까요?

-> 결과: 메인 체인에서 치명적인 에러가 났지만 서비스가 죽지 않고 백업 체인으로 부드럽게 넘어갔습니다.


## 10. 실패/주의 케이스: 타입 불일치 에러
파이프(`|`)로 연결될 때, 앞 단계의 출력 타입과 뒷 단계의 입력 타입이 맞지 않으면 체인이 붕괴됩니다.

In [8]:
print("[타입 불일치 에러 시연]")
# PromptTemplate은 딕셔너리(dict)를 입력으로 받아 문자열(PromptValue)을 출력합니다.
prompt_error = PromptTemplate.from_template("{word}의 뜻은?")

try:
    # 의도적인 실수: 딕셔너리가 아니라 생 문자열을 넘김
    # prompt_error.invoke({"word": "Apple"}) 이 맞습니다.
    chain_error = prompt_error | llm
    chain_error.invoke("Apple")
except Exception as e:
    print("🔥 에러 발생:", type(e).__name__)
    print("에러 메시지:", str(e)[:200] + "...")
    print("\n-> 주의: LCEL 체인을 설계할 때는 항상 이전 노드의 '출력 타입'과 다음 노드의 '입력 타입'이 호환되는지(예: dict -> PromptValue -> AIMessage -> String) 머릿속으로 그려야 합니다.")

[타입 불일치 에러 시연]


## 11. 결과 해석

1. **선언적 프로그래밍**: LCEL은 '어떻게 할 것인가(How)'보다 '무엇을 할 것인가(What)'에 집중하게 해주어 코드 가독성을 극한으로 끌어올립니다.
2. **병렬성 및 유연성**: `RunnableParallel`과 `RunnablePassthrough`를 조합하면 아무리 복잡한 데이터 분기/병합 로직(RAG, Multi-Agent 등)도 깔끔하게 구현할 수 있습니다.
3. **운영 안정성**: 실무에서는 모델 API가 죽거나 제한(Rate Limit)에 걸리는 일이 빈번합니다. `.with_fallbacks()`는 서비스를 무중단으로 유지하기 위한 필수 무기입니다.

## 12. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* 기존에는 변수를 하나하나 할당하느라 코드가 지저분했지만, 파이프(`|`) 연산자를 쓰니 로직이 한 줄로 정리됨.
* LCEL이 강력하긴 하지만, 체인 중간에 타입 에러가 났을 때 디버깅이 약간 까다로움. 각 파이프의 입출력 데이터 규격(Schema)을 정확히 인지해야 함.

### 다음 개선 방향
* `RunnablePassthrough`의 맛보기를 보았으니, 다음 단계에서는 본격적으로 Vector DB에서 실제 문서를 긁어와 체인에 주입하는 **RAG (Retrieval-Augmented Generation)** 파이프라인을 구축해 볼 차례임.